<a href="https://colab.research.google.com/github/BahruzHuseynov/Portfolio/blob/main/Deep_Learning_Projects/CNN_Architecture/CNN2_AlexNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AlexNet

<img src = "https://www.oreilly.com/api/v2/epubs/9781789956177/files/assets/ec08175c-5282-4be2-b6e7-6f2d99272166.png">

<img src = "https://i0.wp.com/syncedreview.com/wp-content/uploads/2017/05/13.png?resize=330%2C230&ssl=1">

In [1]:
import torch
import torchvision
import torchvision.transforms as transforms

from torch.utils.data import DataLoader

from torch import nn
import torch.nn.functional as F
from torch import optim

In [2]:
class AlexNet(nn.Module):
    def __init__(self, num_classes, in_channels = 1):
        super(AlexNet, self).__init__()
        self.num_classes = num_classes
        self.layers = ["conv","maxpool", "conv", "maxpool", "conv", "conv", "conv", "maxpool"]
        self.out_channels = [96, 96, 256, 256, 384, 384, 256, 256]
        self.strides = [4, 2, 1, 2, 1, 1, 1, 2]
        self.pads = [0, 0, 2, 0, 1, 1, 1, 0]
        self.kernels = [11, 3, 5, 3, 3, 3, 3, 3]

        self.seq = nn.Sequential()

        layer = 0
        while layer < len(self.layers):
            if self.layers[layer] == "conv":
                input_channel = in_channels
                if layer != 0:
                    input_channel = self.out_channels[layer-1]

                new_layer = nn.Conv2d(in_channels = input_channel,  out_channels = self.out_channels[layer],
                                      kernel_size = self.kernels[layer], stride = self.strides[layer], padding = self.pads[layer])

                self.seq.add_module(f'layer-{layer}', new_layer)
                self.seq.add_module(f'layer-{layer}-RELU', nn.ReLU())

            if self.layers[layer] == "maxpool":
                new_layer = nn.Conv2d(in_channels = self.out_channels[layer-1], out_channels = self.out_channels[layer],
                                      kernel_size = self.kernels[layer], stride = self.strides[layer], padding = self.pads[layer])

                self.seq.add_module(f'layer-{layer}', new_layer)

            layer += 1

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256*6*6, 4096),
            nn.ReLU(),
            nn.Linear(4096, 4096),
            nn.ReLU(),
            nn.Linear(4096, self.num_classes)
        )

    def forward(self, x):
        x = self.seq(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

In [3]:
model = AlexNet(10)
print(model)

AlexNet(
  (seq): Sequential(
    (layer-0): Conv2d(1, 96, kernel_size=(11, 11), stride=(4, 4))
    (layer-0-RELU): ReLU()
    (layer-1): Conv2d(96, 96, kernel_size=(3, 3), stride=(2, 2))
    (layer-2): Conv2d(96, 256, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (layer-2-RELU): ReLU()
    (layer-3): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2))
    (layer-4): Conv2d(256, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (layer-4-RELU): ReLU()
    (layer-5): Conv2d(384, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (layer-5-RELU): ReLU()
    (layer-6): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (layer-6-RELU): ReLU()
    (layer-7): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2))
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=9216, out_features=4096, bias=True)
    (2): ReLU()
    (3): Linear(in_features=4096, out_features=4096, bias=True)
    (4): ReLU()
    (